In [ ]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "retail_analytics"

orders = spark.table(f"{CATALOG}.{SCHEMA}.silver_orders")
products = spark.table(f"{CATALOG}.{SCHEMA}.silver_products")

In [ ]:
# Enriquecer las órdenes con atributos del producto.
sales = (
    orders.join(products, on="product_id", how="left")
    .withColumn("sales_amount", F.round(F.col("quantity") * F.col("unit_price"), 2))
)

gold_sales = (
    sales
    .groupBy("order_date", "product_id", "product_name", "category")
    .agg(
        F.countDistinct("order_id").alias("orders_count"),
        F.sum("quantity").alias("units_sold"),
        F.round(F.sum("sales_amount"), 2).alias("sales_amount")
    )
)

display(gold_sales.orderBy("order_date", "product_id"))

In [ ]:
(
    gold_sales.write.format("delta").mode("overwrite")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.gold_sales")
)

print("Gold rows:", spark.table(f"{CATALOG}.{SCHEMA}.gold_sales").count())
print("Total sales:", spark.table(f"{CATALOG}.{SCHEMA}.gold_sales")
      .agg(F.round(F.sum("sales_amount"), 2).alias("total_sales"))
      .first()["total_sales"])

In [ ]:
# Indicadores agregados por categoría.
display(
    spark.table(f"{CATALOG}.{SCHEMA}.gold_sales")
    .groupBy("category")
    .agg(
        F.sum("orders_count").alias("orders_count_by_product"),
        F.sum("units_sold").alias("units_sold"),
        F.round(F.sum("sales_amount"), 2).alias("sales_amount")
    )
    .orderBy(F.desc("sales_amount"))
)